FES plots expect a rough structure like this. cluster centres are specific to k value and t value

input/
├── md/
│   ├── topology/
│   │   └── test.gro
│   └── equilibrium-xtc/
│       ├── trp_whole_dir0.xtc       # 230 K
│       └── trp_whole_dir19.xtc      # 277 K
│
└── state-model/
    └── t250_k50/
        └── cluster-centers/
            ├── cluster_016.pdb
            ├── cluster_036.pdb
            └── ...

output/
└── free-energy-surface/
    └── fes_230K_277K_ref036_cold016.pdf

In [ ]:
from __future__ import annotations

from pathlib import Path

import MDAnalysis as mda
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from MDAnalysis.analysis import rms
from matplotlib.ticker import AutoMinorLocator


# =============================================================================
# LOCAL DIRECTORIES
# =============================================================================

input_root = Path("./input").resolve()
output_root = Path(
    "./output/free-energy-surface"
).resolve()


# =============================================================================
# ANALYSIS SETTINGS
# =============================================================================

k_choice = 50
tica_lag = 250

# These are microstate IDs from THIS k/t clustering.
ref_state = 36
cold_state = 16

# Match the original FES setup.
fes_bins = 90

start_frame = 20000
stop_frame = None

x_range = (0.0, 8.0)
y_range = (0.0, 8.0)

temperatures = [
    230,
    277,
]

R_GAS_CONSTANT = 0.00831446261815324  # kJ mol^-1 K^-1


# =============================================================================
# DERIVED INPUT PATHS
# =============================================================================

topology_path = (
    input_root
    / "md"
    / "topology"
    / "test.gro"
)

trajectory_paths = {
    230: (
        input_root
        / "md"
        / "equilibrium-xtc"
        / "trp_whole_dir0.xtc"
    ),
    277: (
        input_root
        / "md"
        / "equilibrium-xtc"
        / "trp_whole_dir19.xtc"
    ),
}

cluster_centers_directory = (
    input_root
    / "state-model"
    / f"t{tica_lag}_k{k_choice}"
    / "cluster-centers"
)

ref_pdb_path = (
    cluster_centers_directory
    / f"cluster_{ref_state:03d}.pdb"
)

cold_pdb_path = (
    cluster_centers_directory
    / f"cluster_{cold_state:03d}.pdb"
)


# =============================================================================
# DERIVED OUTPUT PATH
# =============================================================================

fes_output_directory = output_root

fes_output_directory.mkdir(
    parents=True,
    exist_ok=True,
)

output_path = (
    fes_output_directory
    / (
        f"fes_230K_277K_"
        f"ref{ref_state:03d}_"
        f"cold{cold_state:03d}.pdf"
    )
)


# =============================================================================
# VALIDATE SETTINGS
# =============================================================================

if k_choice <= 0:
    raise ValueError(
        f"k_choice must be positive; got {k_choice}."
    )

for state_name, state_index in [
    ("ref_state", ref_state),
    ("cold_state", cold_state),
]:
    if (
        state_index < 0
        or state_index >= k_choice
    ):
        raise ValueError(
            f"{state_name}={state_index} is invalid for "
            f"k_choice={k_choice}. "
            f"Valid states are 0..{k_choice - 1}."
        )

if ref_state == cold_state:
    raise ValueError(
        "ref_state and cold_state must be different."
    )


# =============================================================================
# VALIDATE INPUT FILES
# =============================================================================

required_files = [
    topology_path,
    ref_pdb_path,
    cold_pdb_path,
    *trajectory_paths.values(),
]

missing_files = [
    path
    for path in required_files
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Missing required FES input files:\n"
        + "\n".join(
            f"  {path}"
            for path in missing_files
        )
    )

print(
    "\nFES input configuration"
)

print(
    f"  k_choice       : {k_choice}"
)

print(
    f"  tica_lag       : {tica_lag}"
)

print(
    f"  ref_state      : {ref_state}"
)

print(
    f"  cold_state     : {cold_state}"
)

print(
    f"  topology       : {topology_path}"
)

print(
    f"  ref PDB        : {ref_pdb_path}"
)

print(
    f"  cold PDB       : {cold_pdb_path}"
)

for temperature in temperatures:
    print(
        f"  {temperature} K trajectory: "
        f"{trajectory_paths[temperature]}"
    )


# =============================================================================
# AXIS FORMATTING
# =============================================================================

def format_axes(
    ax,
    minor_ticks_x=2,
    minor_ticks_y=2,
    label_fontsize=10,
    tick_labelsize=10,
    title_fontsize=10,
    stroke_in=0.01,
):
    line_width_pts = (
        stroke_in
        * 72.0
    )

    ax.minorticks_on()

    if ax.get_xscale() == "log":
        ax.xaxis.set_minor_locator(
            ticker.LogLocator(
                subs="auto"
            )
        )
    else:
        ax.xaxis.set_minor_locator(
            AutoMinorLocator(
                minor_ticks_x
            )
        )

    if ax.get_yscale() == "log":
        ax.yaxis.set_minor_locator(
            ticker.LogLocator(
                subs="auto"
            )
        )
    else:
        ax.yaxis.set_minor_locator(
            AutoMinorLocator(
                minor_ticks_y
            )
        )

    for spine in ax.spines.values():
        spine.set_linewidth(
            1.5
        )

        spine.set_color(
            "black"
        )

    ax.tick_params(
        which="major",
        width=line_width_pts,
        length=4.5,
        direction="in",
        top=True,
        bottom=True,
        left=True,
        right=True,
    )

    ax.tick_params(
        which="minor",
        width=line_width_pts * 0.75,
        length=2.25,
        direction="in",
        top=True,
        bottom=True,
        left=True,
        right=True,
    )

    ax.xaxis.label.set_size(
        label_fontsize
    )

    ax.yaxis.label.set_size(
        label_fontsize
    )

    for tick_label in (
        ax.get_xticklabels()
        + ax.get_yticklabels()
    ):
        tick_label.set_fontsize(
            tick_labelsize
        )

    ax.title.set_fontsize(
        title_fontsize
    )


# =============================================================================
# RMSD
# =============================================================================

def calculate_ca_rmsd(
    universe,
    reference,
    *,
    start=None,
    stop=None,
):
    """
    Calculate CA RMSD after optimal structural alignment.
    """

    reference_atoms = reference.select_atoms(
        "name CA"
    )

    mobile_atoms = universe.select_atoms(
        "name CA"
    )

    if len(reference_atoms) != len(mobile_atoms):
        raise ValueError(
            "Reference and trajectory contain different "
            "numbers of CA atoms.\n"
            f"Reference CA atoms: {len(reference_atoms)}\n"
            f"Trajectory CA atoms: {len(mobile_atoms)}"
        )

    calculation = rms.RMSD(
        mobile_atoms,
        reference_atoms,
        select="name CA",
    )

    calculation.run(
        start=start,
        stop=stop,
    )

    if hasattr(
        calculation,
        "results",
    ):
        values = calculation.results.rmsd[
            :,
            2,
        ]
    else:
        values = calculation.rmsd[
            :,
            2,
        ]

    return np.asarray(
        values,
        dtype=float,
    )


# =============================================================================
# FREE ENERGY SURFACE
# =============================================================================

def compute_free_energy_surface(
    data_x,
    data_y,
    *,
    temperature,
    bins,
    x_range,
    y_range,
):
    """
    Construct the FES exactly as in the original implementation.

    Probability for each bin is relative to ALL finite trajectory
    frames, including frames outside the displayed RMSD window.

        F = -RT ln(p)
    """

    x = np.asarray(
        data_x,
        dtype=float,
    )

    y = np.asarray(
        data_y,
        dtype=float,
    )

    if x.shape != y.shape:
        raise ValueError(
            f"RMSD arrays have different shapes: "
            f"{x.shape} and {y.shape}."
        )

    finite = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    x = x[
        finite
    ]

    y = y[
        finite
    ]

    total_frames = int(
        x.size
    )

    if total_frames == 0:
        raise ValueError(
            "No finite RMSD pairs were available "
            "for the FES."
        )

    histogram, x_edges, y_edges = (
        np.histogram2d(
            x,
            y,
            bins=bins,
            range=[
                list(
                    x_range
                ),
                list(
                    y_range
                ),
            ],
        )
    )

    probability = (
        histogram
        / float(
            total_frames
        )
    )

    free_energy = np.full_like(
        probability,
        np.nan,
        dtype=float,
    )

    occupied = (
        probability > 0.0
    )

    kT = (
        R_GAS_CONSTANT
        * float(
            temperature
        )
    )

    free_energy[
        occupied
    ] = (
        -kT
        * np.log(
            probability[
                occupied
            ]
        )
    )

    return {
        "temperature": temperature,
        "F": free_energy,
        "xedges": x_edges,
        "yedges": y_edges,
        "frame_count": total_frames,
    }


# =============================================================================
# SHARED 230/277 K FREE-ENERGY SCALE
# =============================================================================

def shift_pair_to_common_zero(
    fes_a,
    fes_b,
):
    """
    Use one energy zero and one colour scale for both temperatures.

    This matches the original FES script: the global finite minimum
    across the 230 K and 277 K surfaces is subtracted from both.
    """

    finite_values = []

    for fes in [
        fes_a,
        fes_b,
    ]:
        values = fes[
            "F"
        ]

        values = values[
            np.isfinite(
                values
            )
        ]

        if values.size:
            finite_values.append(
                values
            )

    if not finite_values:
        raise ValueError(
            "Neither FES contains finite free-energy values."
        )

    all_values = np.concatenate(
        finite_values
    )

    common_minimum = float(
        np.min(
            all_values
        )
    )

    common_maximum = float(
        np.max(
            all_values
        )
    )

    for fes in [
        fes_a,
        fes_b,
    ]:
        values = fes[
            "F"
        ]

        fes[
            "F"
        ] = np.where(
            np.isfinite(
                values
            ),
            values
            - common_minimum,
            np.nan,
        )

    colour_minimum = 0.0

    colour_maximum = (
        common_maximum
        - common_minimum
    )

    return (
        colour_minimum,
        colour_maximum,
        common_minimum,
    )


# =============================================================================
# LOAD REFERENCE STRUCTURES
# =============================================================================

reference_universe = mda.Universe(
    str(
        ref_pdb_path
    )
)

cold_universe = mda.Universe(
    str(
        cold_pdb_path
    )
)


# =============================================================================
# CALCULATE RMSDS AND FES
# =============================================================================

fes_by_temperature = {}

for temperature in temperatures:
    print(
        f"\nProcessing {temperature} K"
    )

    trajectory_universe = mda.Universe(
        str(
            topology_path
        ),
        str(
            trajectory_paths[
                temperature
            ]
        ),
    )

    print(
        "  calculating RMSD to reference state "
        f"{ref_state}"
    )

    rmsd_to_reference = calculate_ca_rmsd(
        trajectory_universe,
        reference_universe,
        start=start_frame,
        stop=stop_frame,
    )

    print(
        "  calculating RMSD to cold state "
        f"{cold_state}"
    )

    rmsd_to_cold = calculate_ca_rmsd(
        trajectory_universe,
        cold_universe,
        start=start_frame,
        stop=stop_frame,
    )

    if (
        rmsd_to_reference.shape
        != rmsd_to_cold.shape
    ):
        raise RuntimeError(
            f"{temperature} K RMSD arrays do not match: "
            f"{rmsd_to_reference.shape} versus "
            f"{rmsd_to_cold.shape}"
        )

    fes = compute_free_energy_surface(
        rmsd_to_reference,
        rmsd_to_cold,
        temperature=temperature,
        bins=fes_bins,
        x_range=x_range,
        y_range=y_range,
    )

    fes[
        "rmsd_to_reference"
    ] = rmsd_to_reference

    fes[
        "rmsd_to_cold"
    ] = rmsd_to_cold

    fes_by_temperature[
        temperature
    ] = fes

    print(
        f"  frames used: "
        f"{fes['frame_count']}"
    )

    print(
        f"  RMSD to reference: "
        f"{np.nanmin(rmsd_to_reference):.3f}–"
        f"{np.nanmax(rmsd_to_reference):.3f} Å"
    )

    print(
        f"  RMSD to cold: "
        f"{np.nanmin(rmsd_to_cold):.3f}–"
        f"{np.nanmax(rmsd_to_cold):.3f} Å"
    )


# =============================================================================
# COMMON ENERGY ZERO / COLOUR SCALE
# =============================================================================

(
    vmin,
    vmax,
    original_minimum,
) = shift_pair_to_common_zero(
    fes_by_temperature[
        230
    ],
    fes_by_temperature[
        277
    ],
)

print(
    "\nShared FES scale"
)

print(
    f"  original joint minimum: "
    f"{original_minimum:.6f} kJ/mol"
)

print(
    f"  plotted range: "
    f"{vmin:.6f} to "
    f"{vmax:.6f} kJ/mol"
)


# =============================================================================
# PLOT
# =============================================================================

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(
        7.4,
        3.4,
    ),
    sharex=True,
    sharey=True,
)

last_image = None

for axis, temperature in zip(
    axes,
    temperatures,
):
    fes = fes_by_temperature[
        temperature
    ]

    last_image = axis.imshow(
        fes[
            "F"
        ].T,
        origin="lower",
        extent=[
            fes[
                "xedges"
            ][
                0
            ],
            fes[
                "xedges"
            ][
                -1
            ],
            fes[
                "yedges"
            ][
                0
            ],
            fes[
                "yedges"
            ][
                -1
            ],
        ],
        aspect="auto",
        cmap="inferno",
        vmin=vmin,
        vmax=vmax,
    )

    axis.set_xlim(
        *x_range
    )

    axis.set_ylim(
        *y_range
    )

    axis.set_xlabel(
        "CA RMSD to folded reference (Å)"
    )

    axis.set_title(
        f"{temperature} K"
    )

    format_axes(
        axis,
        label_fontsize=9,
        tick_labelsize=8,
        title_fontsize=10,
    )

axes[
    0
].set_ylabel(
    "CA RMSD to cold denatured reference (Å)"
)

colorbar = fig.colorbar(
    last_image,
    ax=axes,
    fraction=0.035,
    pad=0.04,
)

colorbar.set_label(
    "Free Energy (kJ/mol)",
    fontsize=9,
)

colorbar.ax.tick_params(
    labelsize=8
)

fig.suptitle(
    (
        f"k={k_choice}, tICA lag={tica_lag} "
        f"| ref={ref_state}, cold={cold_state}"
    ),
    fontsize=10,
)

fig.savefig(
    output_path,
    format="pdf",
    bbox_inches="tight",
)

print(
    f"\nSaved FES figure to:\n"
    f"  {output_path}"
)

plt.show()